In [2]:
import pandas as pd
from sqlalchemy import create_engine, text, inspect

engine = create_engine('sqlite:///formula1.sqlite')

In [ ]:
# SELECT, FROM, LIMIT
#forma 1: directo con pandas 
query = text("""
SELECT * FROM drivers LIMIT 5;
""")
# Esta consulta me permite saber las columnas cuidando no sobrecargar  
df = pd.read_sql_query(query, engine)

query2 = text("""SELECT COUNT(1) FROM drivers;""")
# Esta consulta me permite saber el total de filas
df_count = pd.read_sql_query(query2, engine)

'''
#forma 2: usando engine y connection con with

with engine.connect() as connection:
    result = connection.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
df.columns
'''

In [8]:
df.columns

Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')

In [ ]:
query3 = text("""SELECT driverId, forename, surname FROM drivers LIMIT 5;""")
df_driver = pd.read_sql_query(query3, engine)

In [11]:
df

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [13]:
df_driver

,driverId,forename,surname
0,1,Lewis,Hamilton
1,2,Nick,Heidfeld
2,3,Nico,Rosberg
3,4,Fernando,Alonso
4,5,Heikki,Kovalainen


In [ ]:
# Para generar un entregable guardamos el DataFrame en un archivo CSV 
df_driver.to_csv('drivers1.csv', index=False, sep=';')
# Siempre especificar el separador para evitar problemas con los datos 

In [14]:
# SELECT, FROM, WHERE
# pregunta: ¿Cuantos pilotos son británicos?
query4 = text("""
SELECT driverId, forename, surname FROM drivers WHERE nationality = 'British';
""")
df_driver_british = pd.read_sql_query(query4, engine)

In [21]:
# Otra forma.. 
query5 = text("""SELECT COUNT(1) FROM drivers WHERE nationality = 'British';
""")
df_driver_british_count = pd.read_sql_query(query5, engine)

In [22]:
df_driver_british_count

,COUNT(1)
0,166


In [ ]:
# Otra forma  
# ¿Quienes son los pilotos australianos?
query = text("""SELECT * FROM drivers""")
df_all_drivers = pd.read_sql_query(query, engine)

df_all_drivers[df_all_drivers.nationality == 'Australian'][['forename', 'surname']]

,forename,surname
16,Mark,Webber
100,David,Brabham
149,Gary,Brabham
177,Alan,Jones
255,Larry,Perkins
265,Brian,McGuire
266,Vern,Schuppan
285,Warwick,Brown
313,Tim,Schenken
337,David,Walker


In [ ]:
# SELECT, FROM, LIMIT
def select_from_limit():
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM drivers LIMIT 5"))
        for row in result:
            print(row)
# SELECT, FROM, WHERE
def select_from_where():
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM drivers WHERE nationality = 'British'"))
        for row in result:
            print(row)  


In [30]:
df_all_drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 861 entries, 0 to 860
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     861 non-null    int64 
 1   driverRef    861 non-null    object
 2   number       861 non-null    object
 3   code         861 non-null    object
 4   forename     861 non-null    object
 5   surname      861 non-null    object
 6   dob          861 non-null    object
 7   nationality  861 non-null    object
 8   url          861 non-null    object
dtypes: int64(1), object(8)
memory usage: 60.7+ KB


In [39]:
# WHERE CON AND, OR, NOT, IN
# ORDER BY

#¿Cuantos corredores son britanicos o franceses?
query = text('''SELECT forename, surname, nationality, dob
                FROM drivers
                --WHERE nationality IN ('British', 'French')
                --WHERE (nationality = 'British' OR nationality = 'French') AND YEAR(dob)<1999
                WHERE (nationality = 'British' OR nationality = 'French') --AND LEFT(dob,4)=1999
               ORDER BY surname
             ''')

df2 = pd.read_sql_query(query, engine)

In [34]:
df2

,forename,surname,nationality,dob
0,George,Abecassis,British,1913-03-21
1,Kenny,Acheson,British,1957-11-27
2,Jack,Aitken,British,1995-09-23
3,Jean,Alesi,French,1964-06-11
4,Philippe,Alliot,French,1954-07-27
...,...,...,...,...
234,Roger,Williamson,British,1948-02-02
235,Justin,Wilson,British,1978-07-31
236,Vic,Wilson,British,1931-04-14
237,Paul,di Resta,British,1986-04-16
